## Imports

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report

## Load data

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")


DATA_RAW_DIR = "/content/drive/MyDrive/DroneAcharaya/data/raw"
ARTIFACTS_DIR = "/content/drive/MyDrive/DroneAcharaya/ml_artifacts/xgboost_classifier"

os.makedirs(ARTIFACTS_DIR, exist_ok=True)

In [ ]:
import gc

_TELEMETRY_NON_NUMERIC = {"run_id", "engine_state", "engine_id", "mission_id", "data_origin"}
_GT_NON_NUMERIC = {"run_id", "sensor_fault_active_cht_c3", "sensor_fault_active_vibration_rms_x_bearing_proxy"}


def _read_csv_f32(path, non_numeric_cols):
    header_cols = pd.read_csv(path, nrows=0).columns
    dtype_map = {c: "float32" for c in header_cols if c not in non_numeric_cols and c != "t"}
    return pd.read_csv(path, dtype=dtype_map)


telemetry_train_df = _read_csv_f32(f"{DATA_RAW_DIR}/telemetry_train.csv", _TELEMETRY_NON_NUMERIC)
gt_train_df = _read_csv_f32(f"{DATA_RAW_DIR}/groundtruth_train.csv", _GT_NON_NUMERIC)
telemetry_val_df = _read_csv_f32(f"{DATA_RAW_DIR}/telemetry_validation.csv", _TELEMETRY_NON_NUMERIC)
gt_val_df = _read_csv_f32(f"{DATA_RAW_DIR}/groundtruth_validation.csv", _GT_NON_NUMERIC)

telemetry_df = pd.concat([telemetry_train_df, telemetry_val_df], ignore_index=True)
gt_df = pd.concat([gt_train_df, gt_val_df], ignore_index=True)
train_run_ids = set(telemetry_train_df["run_id"].unique())

del telemetry_train_df, gt_train_df, telemetry_val_df, gt_val_df
gc.collect()

print(telemetry_df.shape, gt_df.shape)

## Merge

In [ ]:
merged_df = pd.merge(
    telemetry_df,
    gt_df,
    on=["run_id", "t"],
    how="inner",
    validate="one_to_one",
)

assert len(merged_df) == len(telemetry_df), "Row count changed after merge — check join key alignment"

del telemetry_df, gt_df
gc.collect()

print(merged_df.shape)

## Column groups

In [ ]:
SENSOR_COLUMNS = [
    "rpm",
    "torque",
    "power",
    "engine_load",
    "cht_c1",
    "cht_c2",
    "cht_c3",
    "cht_c4",
    "egt_c1",
    "egt_c2",
    "egt_c3",
    "egt_c4",
    "oil_pressure",
    "oil_temperature",
    "fuel_flow",
    "rail_pressure",
    "injection_timing",
    "boost_pressure",
    "map",
    "intake_temperature",
    "air_mass_flow",
    "coolant_temperature",
    "vibration_rms_x",
    "vibration_order_1x",
    "vibration_rms_x_bearing_proxy",
    "vibration_order_1x_bearing_proxy",
    "battery_voltage",
    "battery_current",
    "alternator_power",
    "altitude",
    "ambient_pressure",
    "ambient_temperature",
    "air_density",
    "throttle",
]

SENSOR_FAULT_COLUMNS = [c for c in merged_df.columns if c.startswith("sensor_fault_active_")]
print("Detected sensor-fault columns:", SENSOR_FAULT_COLUMNS)

## Sensor fault class mapping

In [ ]:
SENSOR_FAULT_CLASSES = ["NONE", "BIAS", "DRIFT", "NOISE", "STUCK", "DROPOUT"]
sensor_fault_class_map = {c: i for i, c in enumerate(SENSOR_FAULT_CLASSES)}

for col in SENSOR_FAULT_COLUMNS:
    merged_df[col] = merged_df[col].map(sensor_fault_class_map)

merged_df[SENSOR_FAULT_COLUMNS].head()

## Rolling features

XGBoost is row-level (no memory of sequence), so a single raw sensor value at
one timestamp can't distinguish fault *type* — a STUCK reading and a genuinely
steady healthy reading look identical at one instant. These rolling features
(computed within each `run_id`, never across run boundaries) give the model
the temporal-shape information it needs: rolling std / rate-of-change surfaces
NOISE and DRIFT, deviation-from-rolling-mean surfaces BIAS, and a stuck-run-length
counter directly targets STUCK.

In [ ]:
ROLLING_FAULT_PRONE_COLUMNS = ["cht_c3", "vibration_rms_x_bearing_proxy"]
ROLLING_WINDOW = 10  # seconds, at this dataset's 1Hz export rate


def add_rolling_features(df: pd.DataFrame, cols: list[str], window: int) -> pd.DataFrame:
    df = df.sort_values(["run_id", "t"])
    grouped = df.groupby("run_id", sort=False)

    for col in cols:
        roll = grouped[col].rolling(window=window, min_periods=1)
        roll_mean = roll.mean().reset_index(level=0, drop=True)
        roll_std = roll.std().reset_index(level=0, drop=True)

        df[f"{col}_roll_mean"] = roll_mean
        df[f"{col}_roll_std"] = roll_std.fillna(0.0)
        df[f"{col}_dev_from_roll_mean"] = df[col] - roll_mean
        df[f"{col}_diff"] = grouped[col].diff().fillna(0.0)

        is_same_as_prev = grouped[col].diff().fillna(1.0) == 0.0
        stuck_run_id = (~is_same_as_prev).groupby(df["run_id"]).cumsum()
        df[f"{col}_stuck_run_length"] = df.groupby(["run_id", stuck_run_id]).cumcount() + 1

    return df


merged_df = add_rolling_features(merged_df, ROLLING_FAULT_PRONE_COLUMNS, ROLLING_WINDOW)

ROLLING_FEATURE_COLUMNS = [
    f"{col}_{suffix}"
    for col in ROLLING_FAULT_PRONE_COLUMNS
    for suffix in ["roll_mean", "roll_std", "dev_from_roll_mean", "diff", "stuck_run_length"]
]
print(f"{len(ROLLING_FEATURE_COLUMNS)} rolling features added")

## Train + val split + scaling

In [ ]:
import joblib
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train_mask = merged_df["run_id"].isin(train_run_ids)

NAN_PRONE_SENSOR_COLUMNS = ["vibration_rms_x", "vibration_order_1x", "vibration_rms_x_bearing_proxy"]
MISSING_FLAG_COLUMNS = [f"{col}_missing" for col in NAN_PRONE_SENSOR_COLUMNS]

for col, flag_col in zip(NAN_PRONE_SENSOR_COLUMNS, MISSING_FLAG_COLUMNS):
    merged_df[flag_col] = merged_df[col].isna().astype(np.float32)

SCALE_COLUMNS = SENSOR_COLUMNS + ROLLING_FEATURE_COLUMNS

scaler = StandardScaler()
scaler.fit(merged_df.loc[train_mask, SCALE_COLUMNS])
joblib.dump(scaler, f"{ARTIFACTS_DIR}/scaler.joblib")
merged_df[SCALE_COLUMNS] = scaler.transform(merged_df[SCALE_COLUMNS])

merged_df[NAN_PRONE_SENSOR_COLUMNS] = merged_df[NAN_PRONE_SENSOR_COLUMNS].fillna(0.0)
merged_df[SCALE_COLUMNS] = merged_df[SCALE_COLUMNS].fillna(0.0)

assert not merged_df[SCALE_COLUMNS].isna().any().any(), "NaNs remain in SCALE_COLUMNS after fill"

engine_state_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
engine_state_encoder.fit(merged_df.loc[train_mask, ["engine_state"]])
engine_state_encoded = engine_state_encoder.transform(merged_df[["engine_state"]])
engine_state_col_names = engine_state_encoder.get_feature_names_out(["engine_state"]).tolist()
for i, col_name in enumerate(engine_state_col_names):
    merged_df[col_name] = engine_state_encoded[:, i]
del engine_state_encoded
gc.collect()

joblib.dump(engine_state_encoder, f"{ARTIFACTS_DIR}/engine_state_encoder.joblib")
print("engine_state columns:", engine_state_col_names)

## FEATURE_COLUMNS

In [ ]:
XGB_FEATURE_COLUMNS = SENSOR_COLUMNS + MISSING_FLAG_COLUMNS + ROLLING_FEATURE_COLUMNS + engine_state_col_names

X_train = merged_df.loc[train_mask, XGB_FEATURE_COLUMNS].values
X_val = merged_df.loc[~train_mask, XGB_FEATURE_COLUMNS].values

print(f"{len(XGB_FEATURE_COLUMNS)} input features")
print(f"{len(SENSOR_FAULT_COLUMNS)} target channels, {len(SENSOR_FAULT_CLASSES)} classes each")
print("Train:", X_train.shape, "| Val:", X_val.shape)

## Class weights

In [ ]:
# Both sensor-fault channels are heavily NONE-dominated -- same reasoning as
# the LSTM notebook's class weighting. XGBoost's sample_weight (per-row) is
# used instead of scale_pos_weight, since this is multiclass, not binary.
sensor_fault_class_weights = {}
for col in SENSOR_FAULT_COLUMNS:
    counts = merged_df.loc[train_mask, col].value_counts().reindex(range(len(SENSOR_FAULT_CLASSES)), fill_value=0)
    freq = counts / counts.sum()
    weight = 1.0 / freq.replace(0, np.nan)
    weight = (weight / weight.sum(skipna=True)).fillna(0.0)
    sensor_fault_class_weights[col] = weight.values
    print(f"{col} class weights:", dict(zip(SENSOR_FAULT_CLASSES, weight.round(4))))

## Train per-channel models

In [ ]:
xgb_models = {}

for channel_col in SENSOR_FAULT_COLUMNS:
    y_train_channel = merged_df.loc[train_mask, channel_col].values.astype(int)
    y_val_channel = merged_df.loc[~train_mask, channel_col].values.astype(int)

    class_weights = sensor_fault_class_weights[channel_col]
    sample_weight_train = class_weights[y_train_channel]

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(SENSOR_FAULT_CLASSES),
        eval_metric="mlogloss",
        early_stopping_rounds=20,
    )

    model.fit(
        X_train,
        y_train_channel,
        sample_weight=sample_weight_train,
        eval_set=[(X_val, y_val_channel)],
        verbose=False,
    )

    xgb_models[channel_col] = model
    print(f"Trained model for {channel_col} (best iteration: {model.best_iteration})")

## Evaluation

In [ ]:
for channel_col, model in xgb_models.items():
    y_val_channel = merged_df.loc[~train_mask, channel_col].values.astype(int)
    y_pred = model.predict(X_val)

    print(f"\n=== {channel_col} ===")
    print(
        classification_report(
            y_val_channel,
            y_pred,
            labels=list(range(len(SENSOR_FAULT_CLASSES))),
            target_names=SENSOR_FAULT_CLASSES,
            zero_division=0,
        )
    )

## SHAP

In [ ]:
import shap

# Pick whichever channel matters most to your demo/report -- both channels
# now have trained models (unlike the earlier single-channel batch), so this
# is not automatically "the only channel" anymore.
inspect_channel = SENSOR_FAULT_COLUMNS[0]
inspect_model = xgb_models[inspect_channel]

explainer = shap.TreeExplainer(inspect_model)
shap_values = explainer.shap_values(X_train[:500])

shap.summary_plot(
    shap_values,
    X_train[:500],
    feature_names=XGB_FEATURE_COLUMNS,
    class_names=SENSOR_FAULT_CLASSES,
)

## Save artifacts

In [ ]:
joblib.dump(xgb_models, f"{ARTIFACTS_DIR}/xgb_sensor_fault_models.joblib")
joblib.dump(XGB_FEATURE_COLUMNS, f"{ARTIFACTS_DIR}/xgb_feature_columns.joblib")
joblib.dump(SENSOR_FAULT_CLASSES, f"{ARTIFACTS_DIR}/sensor_fault_classes.joblib")

print(f"Saved {len(xgb_models)} per-channel XGBoost models to {ARTIFACTS_DIR}")